# MGMT298D: Science and Strategy of AI
## Week 3 Assignment - Clustering & Collaborative Filtering
### Application: Netflix Recommendations

---

**Instructions:** Complete the exercises below by filling in the `???` placeholders and answering the questions in the designated cells. Run all code cells in order.

## Setup and Data Loading

We'll analyze Netflix viewing data to:
1. Segment users into groups with similar tastes (Clustering)
2. Recommend movies based on similar users (Collaborative Filtering)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv("https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/refs/heads/main/netflix_ratings.csv")

print(f"Dataset: {df.shape[0]} users, {df.shape[1]-1} movies")
print(f"\nSample movies: {list(df.columns[1:6])}")
df.head()

In [ ]:
# Quick exploration
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of ratings (using first few movies)
sample_ratings = df.iloc[:, 1:6].values.flatten()
sample_ratings = sample_ratings[sample_ratings > 0]  # Remove zeros
axes[0].hist(sample_ratings, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5], edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ratings')
axes[0].set_xticks([1, 2, 3, 4, 5])

# Average rating per movie (top 10)
avg_ratings = df.iloc[:, 1:].replace(0, np.nan).mean().sort_values(ascending=False).head(10)
axes[1].barh(range(10), avg_ratings.values, color='coral')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([m[:25] + '...' if len(m) > 25 else m for m in avg_ratings.index])
axes[1].set_xlabel('Average Rating')
axes[1].set_title('Top 10 Highest Rated Movies')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---

## Part 1: K-Means Clustering

We'll segment users into groups based on their movie preferences.

In [ ]:
# Prepare data for clustering
data_filled = df.fillna(df.mean(numeric_only=True))
X_full = data_filled.drop('user_id', axis=1)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_full)

print(f"Ready for clustering: {X_scaled.shape[0]} users, {X_scaled.shape[1]} movie features")

In [ ]:
# ============================================================
# EXERCISE 1: Find the optimal number of clusters
# ============================================================
# We'll use the elbow method and silhouette score

k_range = range(2, 11)
inertias = []
silhouette_scores = []

print("Testing different numbers of clusters...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(sil_score)
    print(f"  k={k}: Silhouette = {sil_score:.3f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, 'bo-', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia', fontsize=12)
ax1.set_title('Elbow Method\n(Look for the "elbow" where curve bends)', fontsize=14)
ax1.grid(alpha=0.3)

ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2)
ax2.set_xlabel('Number of Clusters (k)', fontsize=12)
ax2.set_ylabel('Silhouette Score', fontsize=12)
ax2.set_title('Silhouette Analysis\n(Higher is better)', fontsize=14)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Q1:** Based on the elbow curve and silhouette scores, what number of clusters (k) would you choose? Explain your reasoning.

*Your answer:*


In [ ]:
# ============================================================
# EXERCISE 2: Apply K-Means with your chosen k
# ============================================================

CHOSEN_K = ???  # <-- Fill in your chosen number of clusters

kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init='auto')
data_filled['cluster'] = kmeans.fit_predict(X_scaled)

# Show cluster sizes
cluster_sizes = data_filled['cluster'].value_counts().sort_index()
print(f"\nCluster Sizes (k={CHOSEN_K}):")
for c, size in cluster_sizes.items():
    print(f"  Cluster {c}: {size} users ({size/len(data_filled)*100:.1f}%)")

In [ ]:
# Analyze cluster preferences
cluster_centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=X_full.columns
)

# Find top movies for each cluster
print("\n=== Top 5 Movies Per Cluster ===")
for c in range(CHOSEN_K):
    top_movies = cluster_centers.iloc[c].nlargest(5)
    print(f"\nCluster {c}:")
    for movie, rating in top_movies.items():
        short_name = movie[:40] + '...' if len(movie) > 40 else movie
        print(f"  {short_name}: {rating:.2f}")

In [ ]:
# Heatmap of cluster preferences (top 15 most variable movies)
movie_variance = X_full.var().nlargest(15)
top_movies_heatmap = movie_variance.index

cluster_prefs = cluster_centers[top_movies_heatmap]

plt.figure(figsize=(14, 6))
sns.heatmap(cluster_prefs, annot=True, cmap='YlOrRd', fmt='.1f',
            cbar_kws={'label': 'Average Rating'})
plt.title(f'Movie Preferences by User Cluster (k={CHOSEN_K})', fontsize=14)
plt.xlabel('Movies')
plt.ylabel('User Cluster')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Q2:** Look at the top movies for each cluster. Can you give each cluster a descriptive name based on their preferences? (e.g., "Action Lovers", "Rom-Com Fans", etc.)

*Your answer:*
- Cluster 0: 
- Cluster 1: 
- (continue for all clusters)


---

## Part 2: Collaborative Filtering

Now we'll build a recommendation system that predicts ratings based on similar users.

In [ ]:
# Convert to user-item matrix format
user_item = data_filled.set_index('user_id').drop('cluster', axis=1)
ratings_long = user_item.stack().reset_index()
ratings_long.columns = ['user_id', 'movie', 'rating']

# Filter out zero ratings (unwatched)
ratings_long = ratings_long[ratings_long['rating'] > 0]

# Train/test split
train_ratings, test_ratings = train_test_split(ratings_long, test_size=0.2, random_state=42)

print(f"Training ratings: {len(train_ratings):,}")
print(f"Test ratings: {len(test_ratings):,}")

In [ ]:
# Build user-item matrix from training data
train_matrix = train_ratings.pivot(index='user_id', columns='movie', values='rating')
train_matrix = train_matrix.reindex(index=user_item.index, columns=user_item.columns).fillna(user_item.mean())

# Compute user similarity using cosine similarity
user_sim = pd.DataFrame(
    cosine_similarity(train_matrix),
    index=train_matrix.index,
    columns=train_matrix.index
)

print("User similarity matrix computed!")
print(f"Shape: {user_sim.shape}")

In [ ]:
# Visualize user similarity (sample)
sample_users = user_sim.index[:20]
sample_sim = user_sim.loc[sample_users, sample_users]

plt.figure(figsize=(10, 8))
sns.heatmap(sample_sim, cmap='Blues', vmin=0, vmax=1)
plt.title('User Similarity Matrix (Sample of 20 Users)', fontsize=14)
plt.xlabel('User ID')
plt.ylabel('User ID')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXERCISE 3: Tune the number of neighbors (k) for recommendations
# ============================================================

def predict_rating(user_id, movie, k_neighbors):
    """Predict rating using k nearest neighbors."""
    # Get k most similar users
    sims = user_sim[user_id].drop(user_id).nlargest(k_neighbors)
    # Get their ratings for this movie
    neighbor_ratings = train_matrix.loc[sims.index, movie]
    # Weighted average
    return np.dot(sims, neighbor_ratings) / sims.sum()

# Try different values of k. Try: 5, 10, 20, 50, 100
K_NEIGHBORS = ???  # <-- Fill in a value

# Evaluate on test set (sample for speed)
test_sample = test_ratings.sample(min(500, len(test_ratings)), random_state=42)

predictions = []
for _, row in test_sample.iterrows():
    pred = predict_rating(row['user_id'], row['movie'], K_NEIGHBORS)
    predictions.append(pred)

test_sample['predicted'] = predictions
mae = np.abs(test_sample['rating'] - test_sample['predicted']).mean()

print(f"Collaborative Filtering (k={K_NEIGHBORS} neighbors)")
print(f"Test MAE: {mae:.3f}")

In [ ]:
# Explore effect of k
k_values = [5, 10, 20, 50, 100]
maes = []

test_sample_small = test_ratings.sample(200, random_state=42)

for k in k_values:
    preds = [predict_rating(r['user_id'], r['movie'], k) for _, r in test_sample_small.iterrows()]
    mae = np.abs(test_sample_small['rating'] - preds).mean()
    maes.append(mae)
    print(f"k={k}: MAE = {mae:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(k_values, maes, 'go-', linewidth=2, markersize=10)
plt.xlabel('Number of Neighbors (k)', fontsize=12)
plt.ylabel('Test MAE', fontsize=12)
plt.title('Collaborative Filtering: Effect of Neighborhood Size', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

**Q3:** What happens to prediction accuracy (MAE) as you increase the number of neighbors? Is there a trade-off? What's the business implication of choosing k too small vs. too large?

*Your answer:*


In [ ]:
# Show example recommendations for a specific user
EXAMPLE_USER = data_filled['user_id'].iloc[0]

# Find movies this user hasn't rated highly
user_ratings = user_item.loc[EXAMPLE_USER]
unwatched_or_low = user_ratings[user_ratings < 3].index

# Predict ratings for unwatched movies
predicted_ratings = []
for movie in unwatched_or_low[:20]:  # Limit for speed
    pred = predict_rating(EXAMPLE_USER, movie, 20)
    predicted_ratings.append((movie, pred))

# Sort by predicted rating
recommendations = sorted(predicted_ratings, key=lambda x: x[1], reverse=True)[:5]

print(f"\n=== Top 5 Recommendations for User {EXAMPLE_USER} ===")
for movie, rating in recommendations:
    short_name = movie[:45] + '...' if len(movie) > 45 else movie
    print(f"  {short_name}: Predicted Rating {rating:.2f}")

---

## Question 4: Clustering vs. Collaborative Filtering

**Q4:** We used two different techniques: K-Means Clustering and Collaborative Filtering. How are they different? When would you use each one?

| Technique | What it does | Best use case |
|-----------|--------------|---------------|
| K-Means Clustering | | |
| Collaborative Filtering | | |

*Your answer:*


---

## Question 5: Business Applications

**Q5a:** Netflix has identified 4 user segments from clustering. As a marketing manager, how would you use this information to improve the business? Give two specific examples.

*Your answer:*


**Q5b:** Collaborative filtering has a "cold start" problem: it can't recommend movies to new users who haven't rated anything yet. How might Netflix solve this problem?

*Your answer:*


---

## Question 6: Ethical Considerations

**Q6:** Recommendation systems can create "filter bubbles" where users only see content similar to what they've already watched. What are the potential downsides of this for Netflix and its users? How might you design a system to balance personalization with discovery?

*Your answer:*
